In [6]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.auto import tqdm
from google.oauth2 import service_account
import numpy as np

# 1. 신분증(JSON 키) 경로 지정
KEY_PATH = 'google_key.json'

# 2. 인증 객체 생성
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. 프로젝트 ID 설정
project_id = 'coral-bucksaw-482803-p2'

# 4. 데이터 불러오기
query = "SELECT SQLDATE FROM `gdelt-bq.full.events` LIMIT 5"
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("인증 성공! 데이터를 가져왔습니다.")

Downloading: 100%|██████████|
인증 성공! 데이터를 가져왔습니다.


In [10]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    EventCode,
    GoldsteinScale, 
    NumMentions, 
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' OR Actor2CountryCode = 'CHN') AND
    (Actor1CountryCode = 'TWN' OR Actor2CountryCode = 'TWN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 저장
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str).str[:8], format='%Y%m%d')
print(df['SQLDATE'].head())
df.to_csv("project/01_data/raw/gdelt_raw_exp.csv", index=False)
print(f"저장 완료: {len(df)}행")
display(df.head())


ValueError: time data "1970-01-" doesn't match format "%Y%m%d", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [8]:
print(df['SQLDATE'].dtype)

datetime64[ns]


In [9]:
print(df['SQLDATE'].head())


0   1970-01-01 00:00:00.020260530
1   1970-01-01 00:00:00.020260530
2   1970-01-01 00:00:00.020260530
3   1970-01-01 00:00:00.020260530
4   1970-01-01 00:00:00.020260530
Name: SQLDATE, dtype: datetime64[ns]


In [3]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d')

df.to_csv("project/01_data/raw/gdelt_raw.csv", index=False)
print(f"저장 완료: {len(df)}행")

저장 완료: 11967행


In [4]:
import pandas as pd

df = pd.read_csv("project/01_data/processed/final_priority.csv")

print(df[['ActionGeo_Lat', 'ActionGeo_Long']].describe())
print(f"\n좌표 결측치: {df[['ActionGeo_Lat', 'ActionGeo_Long']].isnull().sum().to_dict()}")
print(f"\n좌표 샘플:\n{df[['ActionGeo_Lat', 'ActionGeo_Long', 'priority_score']].head(10)}")

       ActionGeo_Lat  ActionGeo_Long
count    1006.000000     1006.000000
mean       32.434300      114.752113
std         8.297896       27.646586
min       -35.283300     -157.858000
25%        25.047800      116.388000
50%        37.774900      116.388000
75%        39.928900      121.532000
max        55.752200      149.217000

좌표 결측치: {'ActionGeo_Lat': 3, 'ActionGeo_Long': 3}

좌표 샘플:
   ActionGeo_Lat  ActionGeo_Long  priority_score
0            NaN             NaN             NaN
1        24.0000         119.000        0.778261
2        39.9289         116.388        0.704366
3        24.0000         119.000        0.680430
4        24.0000         119.000        0.589327
5        39.9289         116.388        0.561385
6        24.4367         118.318        0.527778
7        39.9289         116.388        0.520000
8        39.9289         116.388        0.518948
9        39.9289         116.388        0.510521


In [5]:
import pandas as pd
df = pd.read_csv("project/01_data/processed/spike_events.csv")
print(df.columns.tolist())
print(df.head(3))

['SQLDATE', 'DailyMentions', 'EventCount', 'AvgGoldstein', 'AvgTone', 'MA_7', 'MA_14', 'MA_30', 'MoM_rate', 'is_spike']
      SQLDATE  DailyMentions  EventCount  AvgGoldstein   AvgTone        MA_7  \
0  2018-04-19            457           5         -7.76 -2.213559  214.428571   
1  2019-01-15            868           3        -10.00 -1.978939  224.000000   
2  2019-01-16           1354           3        -10.00 -2.558135  400.285714   

        MA_14  MA_30     MoM_rate  is_spike  
0  344.642857  452.1   315.454545      True  
1  218.500000  299.9   623.333333      True  
2  283.785714  314.0  1028.333333      True  


In [6]:
import pandas as pd
df = pd.read_csv("project/01_data/processed/final_priority_geo.csv")
print(df.columns.tolist())

['SQLDATE', 'EventCode', 'GoldsteinScale', 'NumMentions', 'AvgTone', 'ActionGeo_Type', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL', 'score_mentions', 'score_goldstein', 'score_tone', 'score_geo', 'priority_score', 'geo_level']


In [7]:
import yaml

with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(config['priority_score']['geo_distance_bins'])

[{'max_km': 22.2, 'score': 1.0}, {'max_km': 88.9, 'score': 0.75}, {'max_km': 177.8, 'score': 0.5}, {'max_km': 355.6, 'score': 0.25}, {'max_km': 99999, 'score': 0.1}]


In [8]:
import pandas as pd

df = pd.read_csv("project/01_data/processed/final_priority.csv")
print(df['score_geo'].value_counts())
print(f"\nScore 범위: {df['priority_score'].min():.3f} ~ {df['priority_score'].max():.3f}")

score_geo
0.10    598
0.50    294
1.00     60
0.25     28
0.75     26
Name: count, dtype: int64

Score 범위: 0.020 ~ 0.989


In [10]:
import pandas as pd
from datetime import datetime, timezone, timedelta

df = pd.read_csv("project/01_data/processed/final_priority_geo.csv")
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])
one_year_ago = datetime.now(timezone.utc) - timedelta(days=365)
df_recent = df[df['SQLDATE'] >= one_year_ago.strftime('%Y-%m-%d')]
print(f"1년 이내 이벤트: {len(df_recent)}개")

FileNotFoundError: [Errno 2] No such file or directory: 'project/01_data/processed/final_priority_geo.csv'

In [ ]:
import pandas as pd

df = pd.read_csv("01_data/processed/satellite_passes.csv")
print(f"전체: {len(df)}건")
print(f"\n이벤트별 근접 위성 수:")
print(df.groupby('SQLDATE')['satellite_name'].count().sort_values(ascending=False))
print(f"\n최근접 거리 분포:")
print(df['min_dist_km'].describe())
print(f"\n샘플:")
print(df[['SQLDATE', 'event_lat', 'event_lon', 'satellite_name', 'min_dist_km']].head(10))

In [ ]:
import pandas as pd
df = pd.read_csv("project/01_data/processed/final_priority_geo.csv")
print(df['SQLDATE'].min())
print(df['SQLDATE'].max())

In [ ]:
import requests
from datetime import datetime
import pandas as pd

def get_cloud_cover(lat, lon, date_str):
    try:
        today = datetime.now().date()
        event_date = pd.to_datetime(date_str).date()

        if event_date < today:
            url = "https://archive-api.open-meteo.com/v1/archive"
            params = {
                "latitude": lat,
                "longitude": lon,
                "start_date": date_str,
                "end_date": date_str,
                "daily": ["cloud_cover_mean"],
                "timezone": "Asia/Tokyo"
            }
        else:
            url = "https://api.open-meteo.com/v1/forecast"
            params = {
                "latitude": lat,
                "longitude": lon,
                "current": ["cloud_cover"],
                "timezone": "Asia/Tokyo"
            }

        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if event_date < today:
            cloud = data['daily']['cloud_cover_mean'][0]
        else:
            cloud = data['current']['cloud_cover']

        return int(cloud) if cloud is not None else None
    except Exception as e:
        print(f"에러: {e}")
        return None

# 테스트
cloud = get_cloud_cover(24.0, 119.0, "2022-08-07")
print(f"구름량: {cloud}%")